In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import csv
import sys
import spacy
from joblib import Parallel, delayed
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

#global variables 
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "tagger", "lemmatizer"])
STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()

URL_RE    = re.compile(r"\bhttps?://[^\s]+|www\.[^\s]+")
EMAIL_RE  = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}")
DATE_RE   = re.compile(r"\b(\d{4}-\d{2}-\d{2}|\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b")
NUM_RE    = re.compile(r"\d+")


#data Loading

def load_data(path):
    """Load CSV from a URL or filepath, handling large field sizes."""
    csv.field_size_limit(sys.maxsize)
    return pd.read_csv(path, encoding="utf-8", engine="python")


#cleaning 

def clean_content(text):
    """
    Clean a single string: lowercase, strip, replace URLs/emails/dates/numbers.
    Operates on a single string rather than a whole DataFrame for efficiency.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = URL_RE.sub("<URL>", text)
    text = EMAIL_RE.sub("<EMAIL>", text)
    text = DATE_RE.sub("<DATE>", text)
    text = NUM_RE.sub("<NUM>", text)
    return text

def clean_corpus(df, column="content"):
    """Apply cleaning to a single column only, preserving all other columns."""
    df = df.copy()
    df[column] = df[column].apply(clean_content)
    return df


#tokenization

def tokenize(text):
    """Tokenize a single string."""
    if not isinstance(text, str) or text == "":
        return []
    return word_tokenize(text)

def tokenize_corpus(df, column="content"):
    df = df.copy()
    texts = df[column].fillna("").tolist()
    df["tokens"] = [
        [token.text for token in doc]
        for doc in nlp.pipe(texts, batch_size=512, n_process=-1)
    ]
    return df


#remove stopwords

def remove_stopwords(tokens):
    """Remove stopwords from a list of tokens."""
    return [t for t in tokens if t.lower() not in STOP_WORDS]

def remove_stopwords_corpus(df, column="tokens"):
    """Apply stopword removal per article."""
    df = df.copy()
    df["tokens_nostop"] = df[column].apply(remove_stopwords)
    return df


#stemming

def stem_tokens(tokens):
    """Apply Porter stemming to a list of tokens."""
    return [STEMMER.stem(t) for t in tokens]

def stem_corpus(df, column="tokens_nostop"):
    """Apply stemming per article."""
    df = df.copy()
    df["tokens_stemmed"] = df[column].apply(stem_tokens)
    return df


#vocab stats

def vocab_stats(df):
    """Report vocabulary size at each processing stage."""
    raw      = set(t for tokens in df["tokens"]         for t in tokens)
    nostop   = set(t for tokens in df["tokens_nostop"]  for t in tokens)
    stemmed  = set(t for tokens in df["tokens_stemmed"] for t in tokens)

    total_tokens = sum(len(t) for t in df["tokens"])

    print(f"Total token count:                {total_tokens:,}")
    print(f"Vocabulary (raw):                 {len(raw):,}")
    print(f"Vocabulary (no stopwords):        {len(nostop):,}  "
          f"({(1 - len(nostop)/len(raw))*100:.2f}% reduction)")
    print(f"Vocabulary (stemmed):             {len(stemmed):,}  "
          f"({(1 - len(stemmed)/len(nostop))*100:.2f}% reduction)")


#train, validation,  test Split

def split_data(df, train=0.8, val=0.1, test=0.1, seed=42):
    """
    Split DataFrame into train, validation, and test sets.
    Returns three DataFrames in order: train, val, test.
    """
    assert round(train + val + test, 10) == 1.0, "Splits must sum to 1."
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df)
    train_end = int(n * train)
    val_end   = int(n * (train + val))
    return df[:train_end], df[train_end:val_end], df[val_end:]


#pipeline

def process_corpus(path, column="content"):
    """
    Full pipeline: load → clean → tokenize → remove stopwords → stem → report.
    Returns the processed DataFrame and the three data splits.
    """
    print("Loading data...")
    df = load_data(path)

    print("Cleaning...")
    df = clean_corpus(df, column)

    print("Tokenizing...")
    df = tokenize_corpus(df, column)

    print("Removing stopwords...")
    df = remove_stopwords_corpus(df)

    print("Stemming...")
    df = stem_corpus(df)

    print("\n Vocabulary Stats:")
    vocab_stats(df)

    print("\nSplitting data...")
    train_df, val_df, test_df = split_data(df)
    print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

    return df, train_df, val_df, test_df

In [2]:
#task 1
URL = "https://raw.githubusercontent.com/several27/FakeNewsCorpus/master/news_sample.csv"
df, train_df, val_df, test_df = process_corpus(URL)

Loading data...
Cleaning...
Tokenizing...
Removing stopwords...
Stemming...

 Vocabulary Stats:
Total token count:                208,899
Vocabulary (raw):                 15,681
Vocabulary (no stopwords):        15,548  (0.85% reduction)
Vocabulary (stemmed):             10,310  (33.69% reduction)

Splitting data...
Train: 200  Val: 25  Test: 25


In [ ]:
#task 2
path = '995,000_rows.csv'
df, train_df, val_df, test_df = process_corpus(path)

Loading data...
Cleaning...
